# MS COCO Multi-label Classification with ViT-Large at 384 x 384 (Google Colab)

This notebook fine-tunes a pretrained **ViT-Large/16** at an input resolution of **384 x 384** on a reduced MS COCO dataset. The model predicts **which of the 80 object categories are present in an image** (multi-label classification). After training, the notebook writes the JSON file expected by the challenge leaderboard.

## Current experiment

| Setting | Value |
|---|---|
| Backbone | `vit_large_patch16_224.augreg_in21k_ft_in1k` (pretrained on ImageNet-21k, fine-tuned on ImageNet-1k; **not** pretrained on COCO) |
| Fine-tuning resolution | 384 x 384 (the 224 x 224 position embeddings are interpolated to the 24 x 24 patch grid) |
| Train / validation split | 80 % / 20 % (seed 42) |
| Batch size / epochs | 8 / 8 |
| Loss | BCE with logits, per-class positive weights, binary label smoothing |
| Optimiser | AdamW, layer-wise learning-rate decay 0.80, peak LR 1e-4, one-cycle schedule |
| Extras | mixed precision, gradient clipping, EMA of the weights (raw and EMA weights are both evaluated) |
| Decision rule | one probability threshold per class, tuned on the validation set (macro-F1) |

## Things to keep in mind

* **Cost.** 384 x 384 gives 576 patch tokens instead of 196 at 224 x 224, and ViT-Large is already several times more expensive than ViT-Base. Expect long epochs; the optional switches in section 2 (resume, mixed-precision evaluation, gradient checkpointing) are there for that reason.
* **Image detail.** The images of this dataset are stored with a long side of 224 px. A 384 px input is therefore an *up-sampled* version: it gives the transformer a finer token grid (which can help small objects) but adds no new image detail.
* **Validation split.** With a 20 % validation set the split differs from experiments that used 12 %. Because the same seed and permutation are used, the 20 % validation set contains the former 12 % validation images *plus* images that older checkpoints were trained on. Do not compare validation scores of older checkpoints with this run on that set.

## How to run

1. `Runtime > Change runtime type > GPU` (a large-memory GPU is recommended for ViT-Large at 384 px).
2. Put the dataset on Google Drive (`ms-coco.zip` recommended) and adjust the paths in section 1.
3. Run sections 1-4 top to bottom, then section 5 (training), then section 6 (submission).
4. After a runtime reset, run sections 1-4 again; to create a submission from an existing checkpoint you can go straight to section 6 (the training cells are not needed).

## 1. Colab environment setup

Reading tens of thousands of small images directly from Google Drive is slow, and the Drive mount can drop during long reads. The cell below copies the dataset **once** to the local disk of the Colab VM (`/content/data`):

* a `.zip` file is copied with a *resumable* copy routine (continues after a dropped connection and verifies the final size) and unzipped locally;
* otherwise the dataset folder is copied file by file;
* a marker file records a successful copy, so re-running the cell is cheap.

In [ ]:
# =============================================================================
# 1. COLAB ENVIRONMENT SETUP
# =============================================================================
# timm provides the pretrained ViT weights and the matching image transforms.
!pip install -q timm

import os
import shutil
import subprocess
import time
from pathlib import Path

import torch
from google.colab import drive

# --- 1.1 GPU check -----------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Use Runtime > Change runtime type > GPU.")

# --- 1.2 Mount Google Drive ---------------------------------------------------
drive.mount('/content/drive')

# --- 1.3 Paths (edit these to match your Drive layout) ------------------------
DRIVE_ZIP  = Path('/content/drive/MyDrive/ms-coco.zip')       # option A: a single zip file (recommended)
DRIVE_DATA = Path('/content/drive/MyDrive/ms-coco')           # option B: an unzipped folder
OUTPUT_DIR = Path('/content/drive/MyDrive/vit_coco_output_v2')  # results are written below this folder (on Drive)
LOCAL_ROOT = Path('/content/data')                            # fast local disk (erased when the runtime resets)


def robust_copy(src, dst, chunk=64 * 1024 * 1024, retries=20):
    """Copy a large file from the (sometimes unstable) Drive mount to local disk.

    * If the connection drops (OSError), the copy continues from the number of
      bytes already written instead of starting over.
    * The final size is verified, so a truncated file can never be used silently.
    """
    src, dst = str(src), str(dst)
    total = os.path.getsize(src)
    for attempt in range(retries + 1):
        done = os.path.getsize(dst) if os.path.exists(dst) else 0
        if done >= total:
            break
        try:
            # Append to the partial destination file and seek the source to the same offset.
            with open(src, 'rb') as fi, open(dst, 'ab' if done else 'wb') as fo:
                fi.seek(done)
                while True:
                    buf = fi.read(chunk)
                    if not buf:
                        break
                    fo.write(buf)
        except OSError as e:
            done = os.path.getsize(dst) if os.path.exists(dst) else 0
            print(f"Connection dropped ({done / 1e9:.2f}/{total / 1e9:.2f} GB), retry {attempt + 1}: {e}")
            time.sleep(5)
    assert os.path.getsize(dst) == total, "Copy is incomplete - please re-run this cell."


# --- 1.4 Drive -> local disk (skipped if it was already done in this session) -
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
done_marker = LOCAL_ROOT / '.copy_done'   # only created after a fully successful copy

if done_marker.exists():
    print("Dataset already present on the local disk - skipping the copy.")
else:
    if DRIVE_ZIP.exists():
        local_zip = Path('/content/ms-coco.zip')
        print("Copying the zip file to the local disk ...")
        robust_copy(DRIVE_ZIP, local_zip)
        print("Unzipping (local) ...")
        # check=True raises an error if unzip fails, so a broken extraction is never marked as done.
        subprocess.run(['unzip', '-q', '-o', str(local_zip), '-d', str(LOCAL_ROOT)], check=True)
        local_zip.unlink()                # free local disk space once extraction succeeded
    elif DRIVE_DATA.exists():
        print(f"Copying the folder (this takes a while): {DRIVE_DATA}")
        shutil.copytree(DRIVE_DATA, LOCAL_ROOT / 'ms-coco', dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            f"Dataset not found on Drive.\n  zip:    {DRIVE_ZIP}\n  folder: {DRIVE_DATA}\nPlease check the paths."
        )
    done_marker.touch()
    print("Copy finished.")

# --- 1.5 Show the directory structure (depth <= 3) to verify the extraction ---
for dirpath, dirnames, _ in os.walk(LOCAL_ROOT):
    depth = len(Path(dirpath).relative_to(LOCAL_ROOT).parts)
    if depth > 3:
        dirnames[:] = []          # do not descend any further
        continue
    print(Path(dirpath).relative_to(LOCAL_ROOT) if depth else '.')

# --- 1.6 Variables used by the rest of the notebook ---------------------------
DATA_ROOT = LOCAL_ROOT                        # dataset_paths() searches below this folder
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Colab VMs have few CPU cores (often 2). More workers than cores only slows data loading down.
NUM_WORKERS = min(8, os.cpu_count() or 2)
print("NUM_WORKERS =", NUM_WORKERS)

## 2. Configuration

All experiment settings live in this cell. The values in the first block define the current experiment. The three **optional switches** at the end are *off by default*, which reproduces the plain training procedure; turn them on only if you need them:

| Switch | Effect | When to use |
|---|---|---|
| `EVAL_USE_AMP` | Run validation and test inference in float16 autocast instead of float32. | Much faster on GPUs with tensor cores (e.g. T4); results can differ in the last decimals. The same setting is used for validation and for test inference so that the tuned thresholds stay consistent. |
| `USE_GRAD_CHECKPOINTING` | Recompute activations in the backward pass. | Saves a lot of GPU memory (use it if training runs out of memory), at roughly +30 % training time. |
| `RESUME_EVERY` | Save a resume state every N epochs and continue from it after a disconnect (0 = disabled). | Long runs on Colab. The resume file of ViT-Large is several GB, so make sure Drive has enough free space. |

In [ ]:
# =============================================================================
# 2. CONFIGURATION
# =============================================================================
from __future__ import annotations

import copy
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from timm.data import create_transform, resolve_data_config
from timm.optim import create_optimizer_v2
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

NUM_CLASSES = 80                       # the 80 MS COCO object categories, class ids 0..79

# ---- Model and input resolution ---------------------------------------------
MODEL_NAME = "vit_large_patch16_224.augreg_in21k_ft_in1k"
RESOLUTION = 384                       # fine-tuning resolution (the pretrained weights were trained at 224)

# ---- Data split and loaders --------------------------------------------------
VALIDATION_FRACTION = 0.20             # 80 % training / 20 % validation
BATCH_SIZE = 8
EPOCHS = 8                             # length of the one-cycle schedule
SEED = 42

# ---- Optimisation hyper-parameters -------------------------------------------
LR = 1e-4                              # peak learning rate of the head / top layers
WEIGHT_DECAY = 0.05
LAYER_DECAY = 0.80                     # layer-wise LR decay: lower layers use a smaller LR
LABEL_SMOOTHING = 0.10                 # epsilon of the binary label smoothing
POS_WEIGHT_STRENGTH = 0.5              # 0 = unweighted loss, 1 = full (capped) negative/positive ratio
DROP_RATE = 0.10                       # dropout before the classification head
EMA_DECAY = 0.9998                     # decay of the weight moving average

# ---- Optional switches (defaults reproduce the plain procedure) --------------
EVAL_USE_AMP = False                   # True: float16 autocast for validation and test inference
USE_GRAD_CHECKPOINTING = False         # True: trade compute for memory during training
RESUME_EVERY = 0                       # N > 0: save a resume state every N epochs (0 = off)

# ---- Output locations (OUTPUT_DIR / NUM_WORKERS come from the setup cell) ----
OUTPUT_DIR_384 = Path(OUTPUT_DIR) / "vit_large_384_current"   # folder dedicated to this experiment
OUTPUT_DIR_384.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT = OUTPUT_DIR_384 / "best_vit_large_384.pt"    # best weights + thresholds + configuration
RESUME_PATH = OUTPUT_DIR_384 / "last_state.pt"                # resume state (only used if RESUME_EVERY > 0)
HISTORY_PATH = OUTPUT_DIR_384 / "training_history.json"       # per-epoch log


def seed_everything(seed: int = 42) -> None:
    """Seed every random number generator so that runs are comparable.

    cudnn.benchmark = True lets cuDNN choose the fastest kernels for the fixed input size;
    this speeds training up but makes it not bit-for-bit deterministic.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

## 3. Data: datasets and path discovery

* **Training images** come with one annotation file per image (`<image id>.cls`) containing the whitespace-separated ids of the categories present in the image. These ids are turned into an 80-dimensional **multi-hot vector** (1 = category present).
* **Test images** have no labels; the dataset returns the image together with its file name (without extension), which is the key used in the submission JSON.
* `dataset_paths` finds the image and label folders for the usual folder layouts.

In [ ]:
# =============================================================================
# 3. DATASETS AND PATH DISCOVERY
# =============================================================================
class CocoTrainDataset(Dataset):
    """Labelled images. Returns (image tensor, 80-dim multi-hot target)."""

    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.transform = transform

        # One <name>.cls file per image; the sorted file stems define a fixed image order.
        self.names = sorted(p.stem for p in self.label_dir.glob("*.cls"))
        if not self.names:
            raise FileNotFoundError(f"No .cls files found in {self.label_dir}")

        # Pre-compute the multi-hot targets once: shape (num_images, 80).
        self.targets = torch.zeros((len(self.names), NUM_CLASSES), dtype=torch.float32)
        for i, name in enumerate(self.names):
            label_file = self.label_dir / f"{name}.cls"
            labels = [int(x) for x in label_file.read_text().split()]
            for cls_id in labels:
                if 0 <= cls_id < NUM_CLASSES:       # ignore ids outside the valid range
                    self.targets[i, cls_id] = 1.0

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = Image.open(self.image_dir / f"{name}.jpg").convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.targets[index]


class CocoTestDataset(Dataset):
    """Unlabelled test images. Returns (image tensor, file stem used as the submission key)."""

    def __init__(self, image_dir, transform):
        self.paths = sorted(Path(image_dir).glob("*.jpg"))
        if not self.paths:
            raise FileNotFoundError(f"No test images found in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = Image.open(path).convert("RGB")
        return self.transform(image), path.stem


def dataset_paths(data_root):
    """Return (train image dir, label dir, test image dir).

    Several folder layouts are tried below `data_root`, `data_root/ms-coco` and
    `data_root/ms_coco`. The first layout whose three folders all exist wins.
    """
    root = Path(data_root)
    candidates = []
    for dataset_root in [root, root / "ms-coco", root / "ms_coco"]:
        candidates.extend([
            # images/train + labels/train + images/test
            (dataset_root / "images" / "train", dataset_root / "labels" / "train", dataset_root / "images" / "test"),
            # images/train + labels (flat) + images/test
            (dataset_root / "images" / "train", dataset_root / "labels", dataset_root / "images" / "test"),
            # COCO-style folder name train2017
            (dataset_root / "images" / "train2017", dataset_root / "labels" / "train", dataset_root / "images" / "test"),
            # train / labels / test directly below the dataset root
            (dataset_root / "train", dataset_root / "labels", dataset_root / "test"),
        ])
    for paths in candidates:
        if all(Path(p).is_dir() for p in paths):
            return paths
    raise FileNotFoundError(f"Could not find dataset below {root}")

## 4. Model, loss weights, metrics, EMA

* `make_model` creates the pretrained network with a new 80-way head. Passing `img_size=RESOLUTION` makes timm **interpolate the pretrained position embeddings** from the 14 x 14 patch grid (224 px) to the 24 x 24 grid (384 px), so the 224 px weights can be fine-tuned at 384 px.
* `make_pos_weight` builds the per-class positive weights of the loss: for a class with `p` positive and `n` negative training images the ratio `n / p` is capped at 20 and blended with 1 according to `strength`. Rare classes get a larger weight on their positive examples.
* `macro_f1` and `calibrate_thresholds` implement the decision rule: a class is predicted when its probability reaches the class threshold, and each class' threshold is the value (0.05 ... 0.85) that maximises that class' F1 on the validation set. **Caveat:** thresholds are tuned on the same images that are used to report the score, so validation scores are slightly optimistic; rare classes have only a few validation positives, which makes their thresholds noisy.
* `EMA` keeps an exponential moving average of the weights; evaluating the averaged weights is usually more stable than evaluating the raw weights.
* `atomic_save` writes through a temporary file so that a disconnect during a save cannot corrupt an existing file.

In [ ]:
# =============================================================================
# 4. MODEL, LOSS WEIGHTS, METRICS, EMA
# =============================================================================
def make_model(model_name, device, resolution=None, drop_rate=None, pretrained=True):
    """Create a timm model with a new 80-way classification head.

    resolution : input size; timm interpolates the pretrained position embeddings to this size.
    pretrained : False when the weights will be overwritten from our own checkpoint (saves a download).
    """
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
        img_size=RESOLUTION if resolution is None else resolution,   # e.g. 384 although the weights are for 224
        drop_rate=DROP_RATE if drop_rate is None else drop_rate,
    )
    return model.to(device)


def make_pos_weight(targets, strength=0.5):
    """Per-class positive weights for BCEWithLogitsLoss. targets: (num_train_images, 80) multi-hot matrix."""
    positives = targets.sum(0).clamp_min(1)                          # avoid division by zero
    ratio = ((len(targets) - positives) / positives).clamp(max=20)   # negatives per positive, capped at 20
    return 1 + strength * (ratio - 1)                                # blend between 1 (no weight) and the ratio


def macro_f1(probs, targets, thresholds):
    """Plain average of the 80 per-class F1 scores. probs/targets: (N, 80), thresholds: (80,)."""
    predictions = probs >= thresholds.unsqueeze(0)       # (N, 80) boolean predictions
    target_bool = targets.bool()
    tp = (predictions & target_bool).sum(0).float()
    fp = (predictions & ~target_bool).sum(0).float()
    fn = (~predictions & target_bool).sum(0).float()
    denom = 2 * tp + fp + fn
    f1 = torch.zeros_like(tp)
    valid = denom > 0                                    # skip classes that neither occur nor get predicted
    f1[valid] = 2 * tp[valid] / denom[valid]
    return f1.mean().item()


def calibrate_thresholds(probs, targets):
    """For every class independently, pick the threshold that maximises that class' F1."""
    grid = torch.arange(0.05, 0.86, 0.01)                # candidate thresholds: 0.05 ... 0.85, step 0.01
    thresholds = torch.full((NUM_CLASSES,), 0.35)        # default for classes without validation positives
    for class_id in range(NUM_CLASSES):
        y = targets[:, class_id].bool()
        if y.sum() == 0:
            continue
        best_f1, best_threshold = -1.0, 0.35
        for threshold in grid:
            pred = probs[:, class_id] >= threshold
            tp = (pred & y).sum().float()
            f1 = (2 * tp / (pred.sum() + y.sum()).clamp_min(1)).item()   # F1 = 2TP / (#predicted + #positives)
            if f1 > best_f1:
                best_f1, best_threshold = f1, float(threshold)
        thresholds[class_id] = best_threshold
    return thresholds, macro_f1(probs, targets, thresholds)


@torch.inference_mode()
def predict_validation(model, loader, device):
    """Run the model over a labelled loader. Returns (probabilities, targets), both on the CPU, shape (N, 80)."""
    model.eval()
    all_probs, all_targets = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        # float32 by default; float16 autocast only if EVAL_USE_AMP is switched on.
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=EVAL_USE_AMP and device.type == "cuda"):
            logits = model(images)
        all_probs.append(logits.float().sigmoid().cpu())   # independent sigmoid per class
        all_targets.append(labels.cpu())
    return torch.cat(all_probs), torch.cat(all_targets)


class EMA:
    """Exponential moving average of the model weights: ema = d * ema + (1 - d) * weights after every step."""

    def __init__(self, model, decay=0.9998):
        self.module = copy.deepcopy(model).eval()        # separate copy that is never trained directly
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.steps = 0

    @torch.no_grad()
    def update(self, model):
        self.steps += 1
        # Warm-up: at the beginning use a smaller decay so the average follows the model quickly.
        decay = min(self.decay, (1 + self.steps) / (10 + self.steps))
        model_state = model.state_dict()
        for key, ema_value in self.module.state_dict().items():
            value = model_state[key].detach()
            if ema_value.dtype.is_floating_point:
                ema_value.mul_(decay).add_(value, alpha=1 - decay)
            else:
                ema_value.copy_(value)                   # integer buffers are copied


def atomic_save(obj, path):
    """torch.save through a temporary file, so an interrupted save never corrupts `path`."""
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

## 5. Training

Section 5 consists of three cells: the **data pipeline**, the **model / optimiser setup**, and the **training loop**. They must be run in this order. Training writes to Google Drive:

* `best_vit_large_384.pt` – the best checkpoint (weights, per-class thresholds, configuration, preprocessing config),
* `training_history.json` – the log of every epoch,
* `last_state.pt` – the resume state (only if `RESUME_EVERY > 0`; removed when training finishes).

### 5.1 Data pipeline

The labelled images are split once into training and validation parts (fixed by the seed). The preprocessing configuration is read from the pretrained model and then **forced to 384 x 384**. Training images use timm's augmentation (random-resized crop, horizontal flip, RandAugment, random erasing); validation images use a deterministic resize + center crop.

In [ ]:
# =============================================================================
# 5.1 DATA PIPELINE
# =============================================================================
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("ViT-Large 384x384 Training")
print("=" * 70)
print(f"Model       : {MODEL_NAME}")
print(f"Resolution  : {RESOLUTION}x{RESOLUTION}")
print(f"Validation  : {VALIDATION_FRACTION * 100:.0f}%")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Epochs      : {EPOCHS}")
print(f"Device      : {device}")
if device.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
print("=" * 70)

image_dir, label_dir, test_dir = dataset_paths(DATA_ROOT)

# ---- Preprocessing configuration ----------------------------------------------
# A temporary pretrained model is built only to read timm's preprocessing defaults
# (normalisation mean/std, interpolation, crop fraction, ...).
probe_model = make_model(MODEL_NAME, device)
data_config = resolve_data_config(probe_model.pretrained_cfg, model=probe_model)
del probe_model
if device.type == "cuda":
    torch.cuda.empty_cache()

# The pretrained config says 224; force the fine-tuning resolution.
data_config["input_size"] = (3, RESOLUTION, RESOLUTION)

# ---- Train / validation split (fixed by the seed) ------------------------------
base_dataset = CocoTrainDataset(image_dir, label_dir)      # used for the targets only (no transform)

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(base_dataset), generator=generator).tolist()
split = int(len(base_dataset) * (1 - VALIDATION_FRACTION))
train_indices = indices[:split]
val_indices = indices[split:]

print("\nDataset")
print("-" * 70)
print(f"Total      : {len(base_dataset)}")
print(f"Train      : {len(train_indices)}")
print(f"Validation : {len(val_indices)}")
print("-" * 70)

# ---- Transforms ------------------------------------------------------------------
train_transform = create_transform(
    **data_config,
    is_training=True,
    auto_augment="rand-m9-mstd0.5-inc1",    # RandAugment: magnitude 9, magnitude noise 0.5
    re_prob=0.25,                           # random erasing with probability 0.25
    re_mode="pixel",
)
val_transform = create_transform(**data_config, is_training=False)   # deterministic resize + center crop

# ---- Datasets: two views of the same files, differing only in the transform ------
train_dataset_full = CocoTrainDataset(image_dir, label_dir, train_transform)
val_dataset_full = CocoTrainDataset(image_dir, label_dir, val_transform)
train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

# ---- Data loaders ------------------------------------------------------------------
loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),   # keep workers alive between epochs
)
train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
print(f"Steps per epoch: {len(train_loader)}")

### 5.2 Model, EMA, loss, optimiser and learning-rate schedule

* **Loss:** `BCEWithLogitsLoss` with per-class positive weights computed from the *training part only*.
* **Optimiser:** AdamW with layer-wise learning-rate decay. timm creates one parameter group per layer depth, each carrying an `lr_scale` (1.0 for the head, smaller for lower layers); bias, normalisation and position-embedding parameters are excluded from weight decay.
* **Schedule:** one-cycle. The learning rate warms up over the first 15 % of the steps to its peak (`lr_scale` times `LR` for each group), then decays smoothly to a very small value. The scheduler is stepped after **every batch**.
* **Mixed precision:** a gradient scaler keeps float16 gradients from underflowing.

In [ ]:
# =============================================================================
# 5.2 MODEL, EMA, LOSS, OPTIMISER, SCHEDULER
# =============================================================================
print("\nLoading ViT-Large...")
model = make_model(MODEL_NAME, device)
if USE_GRAD_CHECKPOINTING:
    model.set_grad_checkpointing(True)      # recompute activations during backward to save memory
print("Model loaded.")

# EMA copy of the weights (updated after every optimiser step).
ema = EMA(model, decay=EMA_DECAY)

# ---- Loss ----------------------------------------------------------------------------
pos_weight = make_pos_weight(base_dataset.targets[train_indices], strength=POS_WEIGHT_STRENGTH).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="mean")

# ---- Optimiser with layer-wise learning-rate decay ------------------------------------
optimizer = create_optimizer_v2(
    model,
    opt="adamw",
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    layer_decay=LAYER_DECAY,
    betas=(0.9, 0.999),
)

# ---- One-cycle learning-rate schedule --------------------------------------------------
# torch's OneCycleLR does not know timm's `lr_scale`, so the peak LR of every group is passed explicitly.
max_lrs = [LR * group.get("lr_scale", 1.0) for group in optimizer.param_groups]
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lrs,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,          # 15 % of the steps are warm-up
    div_factor=25,           # initial LR = peak / 25
    final_div_factor=1000,   # final LR = initial LR / 1000
)

# ---- Mixed-precision gradient scaler ----------------------------------------------------
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

### 5.3 Training loop

For every epoch:

1. **Training pass** – binary label smoothing (`1 -> 1 - eps + eps/80`, `0 -> eps/80`), mixed-precision forward pass, loss, backward pass, gradient clipping at norm 1.0, optimiser / scheduler / EMA updates.
2. **Validation** – the raw weights and the EMA weights are both evaluated; class thresholds are calibrated for each, and the better of the two (higher macro-F1) becomes the candidate of the epoch.
3. **Checkpointing** – if the candidate beats the best score so far, the weights (raw or EMA), thresholds and configuration are saved to Drive.
4. The epoch is appended to `training_history.json`. If `RESUME_EVERY > 0`, a resume state is written every N epochs.

The learning rate that is printed is the **head's** learning rate (the largest of all groups); lower layers use smaller values because of the layer-wise decay.

In [ ]:
# =============================================================================
# 5.3 TRAINING LOOP
# =============================================================================
best_score = -1.0
best_epoch = -1
best_thresholds = None
history = []
start_epoch = 1

# ---- Optional: continue from a resume state after a disconnect ---------------------------
# The state is only used if it was written by exactly this configuration (compared via a signature).
signature = {
    "model_name": MODEL_NAME, "resolution": RESOLUTION, "validation_fraction": VALIDATION_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "seed": SEED, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "layer_decay": LAYER_DECAY, "label_smoothing": LABEL_SMOOTHING,
    "pos_weight_strength": POS_WEIGHT_STRENGTH, "drop_rate": DROP_RATE, "ema_decay": EMA_DECAY,
}
if RESUME_EVERY > 0 and RESUME_PATH.exists():
    state = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    if state.get("signature") == signature:
        model.load_state_dict(state["model"])
        ema.module.load_state_dict(state["ema"])
        ema.steps = state["ema_steps"]
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch = state["epoch"] + 1
        best_score, best_epoch = state["best_score"], state["best_epoch"]
        best_thresholds, history = state["best_thresholds"], state["history"]
        print(f"Resuming after epoch {state['epoch']} (best F1 so far: {best_score:.5f})")
    else:
        print("WARNING: a resume file from a DIFFERENT configuration was found and is ignored.")
    del state

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()

    # ---------------------------------------------------------------------------
    # Training pass
    # ---------------------------------------------------------------------------
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Binary label smoothing: positives 1 -> 1 - eps + eps/80, negatives 0 -> eps/80.
        if LABEL_SMOOTHING > 0:
            labels_for_loss = labels * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / NUM_CLASSES
        else:
            labels_for_loss = labels

        optimizer.zero_grad(set_to_none=True)

        # Mixed-precision forward pass (float16 on GPU; autocast is disabled on CPU).
        with torch.autocast(
            device_type="cuda" if device.type == "cuda" else "cpu",
            dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(images)
            loss = criterion(logits, labels_for_loss)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                    # unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # stabilises training
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()                                              # the schedule advances per batch
        ema.update(model)                                             # update the moving-average weights

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_dataset)

    # ---------------------------------------------------------------------------
    # Validation of the raw weights
    # ---------------------------------------------------------------------------
    raw_probs, val_targets = predict_validation(model, val_loader, device)
    raw_thresholds, raw_score = calibrate_thresholds(raw_probs, val_targets)

    # ---------------------------------------------------------------------------
    # Validation of the EMA weights
    # ---------------------------------------------------------------------------
    ema_probs, _ = predict_validation(ema.module, val_loader, device)
    ema_thresholds, ema_score = calibrate_thresholds(ema_probs, val_targets)

    # ---------------------------------------------------------------------------
    # Keep the better of raw / EMA as the candidate of this epoch
    # ---------------------------------------------------------------------------
    if ema_score >= raw_score:
        epoch_score, epoch_thresholds, selected_model = ema_score, ema_thresholds, "EMA"
    else:
        epoch_score, epoch_thresholds, selected_model = raw_score, raw_thresholds, "RAW"

    # Head learning rate = largest LR over all parameter groups (group 0 is the lowest layer).
    current_lr = max(group["lr"] for group in optimizer.param_groups)

    print(f"\nEpoch {epoch:02d}/{EPOCHS}   ({(time.time() - epoch_start) / 60:.1f} min)")
    print(f"  train_loss : {train_loss:.5f}")
    print(f"  raw F1     : {raw_score:.5f}")
    print(f"  EMA F1     : {ema_score:.5f}")
    print(f"  selected   : {selected_model}")
    print(f"  best F1    : {max(best_score, epoch_score):.5f}")
    print(f"  lr (head)  : {current_lr:.8f}")

    # ---------------------------------------------------------------------------
    # Save the checkpoint if this epoch is the best so far
    # ---------------------------------------------------------------------------
    if epoch_score > best_score:
        best_score = epoch_score
        best_epoch = epoch
        best_thresholds = epoch_thresholds.clone()

        checkpoint_model = ema.module if selected_model == "EMA" else model
        checkpoint = {
            "state_dict": checkpoint_model.state_dict(),   # raw or EMA weights, whichever scored higher
            "thresholds": best_thresholds,                 # per-class thresholds used at inference
            "score": best_score,
            "epoch": best_epoch,
            "model_name": MODEL_NAME,
            "resolution": RESOLUTION,
            "data_config": data_config,                    # preprocessing needed at inference time
            "validation_fraction": VALIDATION_FRACTION,
            "seed": SEED,
            "hyperparameters": {
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "layer_decay": LAYER_DECAY,
                "label_smoothing": LABEL_SMOOTHING,
                "pos_weight_strength": POS_WEIGHT_STRENGTH,
                "drop_rate": DROP_RATE,
            },
        }
        atomic_save(checkpoint, BEST_CHECKPOINT)
        print("  * BEST CHECKPOINT SAVED")
        print(f"    {BEST_CHECKPOINT}")

    # ---------------------------------------------------------------------------
    # Log the epoch (written every epoch so the log survives a disconnect)
    # ---------------------------------------------------------------------------
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "raw_f1": raw_score,
        "ema_f1": ema_score,
        "selected": selected_model,
        "selected_f1": epoch_score,
        "lr": current_lr,
    })
    HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")

    # ---------------------------------------------------------------------------
    # Optional: write the resume state (model, EMA, optimiser, scheduler, scaler, progress)
    # ---------------------------------------------------------------------------
    if RESUME_EVERY > 0 and epoch % RESUME_EVERY == 0 and epoch < EPOCHS:
        atomic_save({
            "signature": signature, "epoch": epoch,
            "best_score": best_score, "best_epoch": best_epoch,
            "best_thresholds": best_thresholds, "history": history,
            "model": model.state_dict(), "ema": ema.module.state_dict(), "ema_steps": ema.steps,
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
        }, RESUME_PATH)

# The run finished, so the resume state is no longer needed.
if RESUME_PATH.exists():
    RESUME_PATH.unlink()

print("\n")
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(f"Best epoch : {best_epoch}")
print(f"Best F1    : {best_score:.5f}")
print(f"Resolution : {RESOLUTION}x{RESOLUTION}")
print(f"Checkpoint : {BEST_CHECKPOINT}")
print("=" * 70)

## 6. Test inference and submission JSON

`create_submission` rebuilds the network from the checkpoint (resolution and dropout are read from it; the weights come from the checkpoint, so the pretrained weights are **not** downloaded again), predicts the test images and writes a JSON file of the form

```json
{"000000000139": [56, 60, 62], "000000000285": [21], ...}
```

i.e. for every test image (key = file name without extension) the list of predicted class ids. `check_submission` verifies the format before the file is uploaded to the leaderboard.

This section only needs the definitions of sections 1-4, not the training cells, so it can be run after a runtime reset to create a submission from an existing checkpoint.

> **Before uploading:** submitting again under the same group name replaces the previous leaderboard entry (see the course instructions). Upload only the checkpoint you want to keep as your final entry.

In [ ]:
# =============================================================================
# 6. TEST INFERENCE AND SUBMISSION FILE
# =============================================================================
@torch.inference_mode()
def create_submission(data_root, checkpoint_path, json_path, *, batch_size=32, num_workers=4):
    """Predict the test images with a saved checkpoint and write the leaderboard JSON."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    # Rebuild the architecture exactly as trained, then load the trained weights.
    model = make_model(
        checkpoint["model_name"], device,
        resolution=checkpoint["resolution"],
        drop_rate=checkpoint["hyperparameters"]["drop_rate"],
        pretrained=False,                       # weights come from the checkpoint
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    _, _, test_dir = dataset_paths(data_root)
    test_transform = create_transform(**checkpoint["data_config"], is_training=False)   # same as validation
    loader = DataLoader(
        CocoTestDataset(test_dir, test_transform),
        batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=device.type == "cuda",
        persistent_workers=num_workers > 0,
    )

    thresholds = checkpoint["thresholds"].to(device)       # per-class thresholds tuned on the validation set
    predictions = {}
    for images, names in loader:
        # Same precision mode as the validation pass (EVAL_USE_AMP), so the thresholds stay consistent.
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=EVAL_USE_AMP and device.type == "cuda"):
            logits = model(images.to(device, non_blocking=True))
        positive = logits.float().sigmoid() >= thresholds   # (batch, 80) booleans
        for name, row in zip(names, positive.cpu()):
            predictions[name] = torch.where(row)[0].tolist()   # indices of the predicted classes

    json_path = Path(json_path)
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(predictions, indent=2))
    print(f"Wrote {len(predictions)} test predictions to {json_path}")
    return json_path


def check_submission(json_path, data_root):
    """Verify the format: one entry per test image, every value a list of ints in [0, 79]."""
    _, _, test_dir = dataset_paths(data_root)
    expected = {p.stem for p in Path(test_dir).glob("*.jpg")}
    predictions = json.loads(Path(json_path).read_text())

    missing = expected - predictions.keys()
    extra = predictions.keys() - expected
    malformed = [
        k for k, v in predictions.items()
        if not (isinstance(v, list) and all(isinstance(i, int) and 0 <= i < NUM_CLASSES for i in v))
    ]
    empty = sum(1 for v in predictions.values() if len(v) == 0)

    print(f"entries: {len(predictions)} | test images: {len(expected)}")
    print(f"missing keys: {len(missing)} | unexpected keys: {len(extra)} | malformed values: {len(malformed)}")
    print(f"images with an empty prediction list: {empty}")
    assert not missing and not extra and not malformed, "Submission format check failed."
    print("Submission format OK.")

In [ ]:
# =============================================================================
# 6.1 CREATE AND CHECK THE SUBMISSION
# =============================================================================
assert BEST_CHECKPOINT.exists(), f"Checkpoint not found: {BEST_CHECKPOINT}"

SUBMISSION_JSON = create_submission(
    data_root=DATA_ROOT,
    checkpoint_path=BEST_CHECKPOINT,
    json_path=OUTPUT_DIR_384 / "submission_vit_large_384.json",
    batch_size=32,                  # lower this on CUDA out-of-memory
    num_workers=NUM_WORKERS,
)
check_submission(SUBMISSION_JSON, DATA_ROOT)   # format check before uploading to the leaderboard

## 7. Experiment log (fill in for the comparative study)

Keep one row per run so that models and settings can be compared on equal terms (same split, same metric, same number of epochs).

| Run | Model | Resolution | Val. split | Epochs | Best val. macro-F1 | Server F1 | Notes |
|---|---|---|---|---|---|---|---|
| this notebook | ViT-Large/16 | 384 | 20 % |  8 |  |  |  |

Points worth discussing in the analysis:

* **Validation vs. server score.** Thresholds are tuned on the validation set, and rare classes have very few validation and test positives, so their F1 is noisy; small differences between runs may not be significant.
* **Resolution.** The source images have a long side of 224 px, so 384 px input is up-sampled: it changes the token grid, not the image content.
* **Raw vs. EMA weights.** The training log prints both for every epoch; the checkpoint stores whichever scored better.